# LeetCode 2128 — Remove All Ones With Row and Column Flips bằng Python

Link đề: https://leetcode.com/problems/remove-all-ones-with-row-and-column-flips/ · Độ khó: **Medium** · Chủ đề: Array, Math, Bit Manipulation, Matrix

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def removeOnes(self, grid: List[List[int]]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa phép "lật" bằng **XOR**: lật hai lần = không lật; thứ tự các phép lật không quan trọng ⇒ mỗi hàng/cột chỉ cần quyết định **lật hay không** (biến nhị phân).
2. Biến bài toán thành hệ phương trình `grid[i][j] = r_i ⊕ c_j` và giải nó bằng cách **cố định một biến** (không mất tổng quát).
3. Rút ra tiêu chí đẹp: mọi hàng phải **bằng hàng 0 hoặc là phần bù của hàng 0**.
4. Cài đặt 3–4 cách: vét cạn tập lật cột, "lật theo hàng 0 rồi kiểm tra", chuẩn hóa hàng + `set`, kiểm tra chẵn lẻ hình chữ nhật.
5. Liên hệ các bài cùng họ (LC 1072, 861) và biến thể "số phép lật tối thiểu".

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` kích thước `m × n`. Mỗi **thao tác**: chọn một hàng **hoặc** một cột và lật mọi giá trị trong đó (0 thành 1, 1 thành 0).

Trả về `True` nếu có thể làm **mọi ô bằng 0** bằng số thao tác tùy ý.

**Ví dụ 1:** `grid = [[0,1,0],[1,0,1],[0,1,0]]` → `True` (lật hàng giữa, rồi lật cột giữa).

**Ví dụ 2:** `grid = [[1,1,0],[0,0,0],[0,0,0]]` → `False`.

**Ví dụ 3:** `grid = [[0]]` → `True` (không có số 1 nào).

**Ràng buộc:** `1 ≤ m, n ≤ 300`, `grid[i][j] ∈ {0, 1}`.

In [1]:
from typing import List
from itertools import product
import random, time

examples = [
    ([[0, 1, 0], [1, 0, 1], [0, 1, 0]], True),
    ([[1, 1, 0], [0, 0, 0], [0, 0, 0]], False),
    ([[0]], True),
]
def show(g):
    for row in g:
        print("   " + " ".join(map(str, row)))

def flip_row(g, i):
    g[i] = [x ^ 1 for x in g[i]]
def flip_col(g, j):
    for row in g:
        row[j] ^= 1

g = [row[:] for row in examples[0][0]]
print("ban đầu:"); show(g)
flip_row(g, 1); print("lật hàng 1:"); show(g)
flip_col(g, 1); print("lật cột 1:"); show(g)

ban đầu:
   0 1 0
   1 0 1
   0 1 0
lật hàng 1:
   0 1 0
   0 1 0
   0 1 0
lật cột 1:
   0 0 0
   0 0 0
   0 0 0


## 3. Kiến thức nền

### 3.1 XOR (`^`) — phép lật bit

`x ^ 1` lật bit `x`; `x ^ 0 = x`. Tính chất: giao hoán, kết hợp, `a ^ a = 0`. Do đó lật một ô `k` lần tương đương lật `k mod 2` lần.

In [2]:
for x in [0, 1]:
    print(f"x={x}: x^1={x ^ 1}, x^1^1={x ^ 1 ^ 1}, x^0={x ^ 0}")
print("lật 5 lần = lật", 5 % 2, "lần:", 0 ^ 1 ^ 1 ^ 1 ^ 1 ^ 1)

x=0: x^1=1, x^1^1=0, x^0=0
x=1: x^1=0, x^1^1=1, x^0=1
lật 5 lần = lật 1 lần: 1


### 3.2 Thứ tự các phép lật không quan trọng

Ô `(i, j)` chỉ bị ảnh hưởng bởi phép lật hàng `i` và phép lật cột `j`. Giá trị cuối = `grid[i][j] ⊕ (số lần lật hàng i) ⊕ (số lần lật cột j)` (mod 2) — không phụ thuộc thứ tự. Ta kiểm chứng bằng cách áp cùng một tập phép lật theo hai thứ tự khác nhau:

In [3]:
ops = [("r", 0), ("c", 2), ("r", 1), ("c", 0), ("r", 0)]
g1 = [row[:] for row in examples[1][0]]; g2 = [row[:] for row in examples[1][0]]
for t, k in ops: (flip_row if t == "r" else flip_col)(g1, k)
for t, k in reversed(ops): (flip_row if t == "r" else flip_col)(g2, k)
print(g1 == g2); show(g1)

True
   0 1 1
   0 1 0
   1 0 1


### 3.3 `tuple` làm phần tử của `set`

`list` không băm được (mutable) nên không cho vào `set`; `tuple` thì được. Dùng `set` của các hàng (dạng tuple) để đếm có bao nhiêu "mẫu hàng" khác nhau.

In [4]:
rows = [[0, 1, 0], [1, 0, 1], [0, 1, 0]]
s = {tuple(r) for r in rows}
print(s, len(s))
try:
    {[0, 1]}
except TypeError as e:
    print("list không băm được:", e)

{(1, 0, 1), (0, 1, 0)} 2
list không băm được: unhashable type: 'list'


### 3.4 `itertools.product([0,1], repeat=k)` — liệt kê mọi vector nhị phân

Dùng cho vét cạn: `2^k` lựa chọn lật/không lật.

In [5]:
print(list(product([0, 1], repeat=3)))

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]


## 4. Ý tưởng nền tảng / trực giác

Từ 3.1–3.2, một "lời giải" được mô tả hoàn toàn bởi hai vector nhị phân: `r = (r_0, …, r_{m−1})` (hàng nào bị lật) và `c = (c_0, …, c_{n−1})` (cột nào bị lật). Yêu cầu:

`grid[i][j] ⊕ r_i ⊕ c_j = 0` ⇔ **`grid[i][j] = r_i ⊕ c_j`** với mọi `i, j`.

Thử tay ví dụ 1: chọn `r = (0, 1, 0)`, `c = (0, 1, 0)`:

| | c=0 | c=1 | c=0 |
|---|---|---|---|
| r=0 | 0 | 1 | 0 |
| r=1 | 1 | 0 | 1 |
| r=0 | 0 | 1 | 0 |

Đúng bằng `grid` ⇒ `True`.

**Quan sát then chốt:** nếu `grid[i][j] = r_i ⊕ c_j` thì hàng `i` = `c ⊕ r_i` — tức **mọi hàng đều bằng `c` hoặc bằng phần bù của `c`**. Hai hàng bất kỳ hoặc **giống hệt**, hoặc **bù nhau**.

Ví dụ 2: hàng 0 = `110`, hàng 1 = `000` — không giống, không bù ⇒ `False`.

In [6]:
def relation(a, b):
    if a == b: return "giống"
    if all(x != y for x, y in zip(a, b)): return "bù"
    return "KHÔNG giống, KHÔNG bù"
for g, exp in examples:
    print("grid =", g, "kỳ vọng", exp)
    for i in range(1, len(g)):
        print(f"   hàng 0 vs hàng {i}: {relation(g[0], g[i])}")

grid = [[0, 1, 0], [1, 0, 1], [0, 1, 0]] kỳ vọng True
   hàng 0 vs hàng 1: bù
   hàng 0 vs hàng 2: giống
grid = [[1, 1, 0], [0, 0, 0], [0, 0, 0]] kỳ vọng False
   hàng 0 vs hàng 1: KHÔNG giống, KHÔNG bù
   hàng 0 vs hàng 2: KHÔNG giống, KHÔNG bù
grid = [[0]] kỳ vọng True


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi tập lật hàng + cột: O(2^(m+n)·m·n)

Đúng theo định nghĩa (sau khi biết mỗi hàng/cột lật tối đa một lần). Chỉ dùng cho ma trận rất nhỏ.

In [7]:
def brute_all(grid):
    m, n = len(grid), len(grid[0])
    for r in product([0, 1], repeat=m):
        for c in product([0, 1], repeat=n):
            if all(grid[i][j] ^ r[i] ^ c[j] == 0 for i in range(m) for j in range(n)):
                return True
    return False

for g, exp in examples:
    print(brute_all(g), "OK" if brute_all(g) == exp else "SAI")

True OK
False OK
True OK


### 5.2 Vét cạn chỉ tập lật cột: O(2^n·m·n)

Khi đã chọn `c`, mỗi hàng chỉ còn một lựa chọn: hàng `i` sau khi lật cột là `grid[i] ⊕ c`; nó phải **đồng nhất** (toàn 0 — không lật hàng, hoặc toàn 1 — lật hàng). Vậy chỉ cần thử `2^n` vector `c`.

**Nút thắt:** vẫn mũ theo `n` (n = 300 ⇒ 2³⁰⁰). Ta cần nhận ra `c` gần như bị **ép** bởi hàng 0.

In [8]:
def brute_cols(grid):
    m, n = len(grid), len(grid[0])
    for c in product([0, 1], repeat=n):
        ok = True
        for row in grid:
            t = [x ^ y for x, y in zip(row, c)]
            if len(set(t)) != 1:            # hàng không đồng nhất -> không lật hàng nào cứu được
                ok = False; break
        if ok:
            return True
    return False

for g, exp in examples:
    print(brute_cols(g), "OK" if brute_cols(g) == exp else "SAI")

True OK
False OK
True OK


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đối xứng `(r, c) ~ (r ⊕ 1, c ⊕ 1)`:** lật **tất cả** hàng và **tất cả** cột thì mỗi ô bị lật đúng 2 lần ⇒ không đổi. Vậy nếu `(r, c)` là lời giải thì `(¬r, ¬c)` cũng là lời giải. Do đó **không mất tổng quát** giả sử `r_0 = 0` (hàng 0 không lật).

**Insight 2 — hàng 0 ép toàn bộ `c`:** với `r_0 = 0`: `grid[0][j] = c_j` ⇒ `c = grid[0]`.

**Insight 3 — cột 0 ép toàn bộ `r`:** `grid[i][0] = r_i ⊕ c_0` ⇒ `r_i = grid[i][0] ⊕ grid[0][0]`.

Khi đã xác định hết `r`, `c`, chỉ cần **kiểm tra** mọi ô:

`grid[i][j] == grid[i][0] ⊕ grid[0][0] ⊕ grid[0][j]` ⇔ `grid[i][j] ⊕ grid[i][0] ⊕ grid[0][j] ⊕ grid[0][0] = 0`.

Diễn giải hình học: **mọi hình chữ nhật** có một góc ở `(0, 0)` phải có **số chẵn** ô bằng 1 ở bốn góc. (Và từ đó suy ra mọi hình chữ nhật bất kỳ cũng vậy.)

Quy tắc "nếu… thì…":
- **Nếu** `grid[i][0] == grid[0][0]` **thì** hàng `i` phải **bằng** hàng 0.
- **Nếu** `grid[i][0] != grid[0][0]` **thì** hàng `i` phải là **phần bù** của hàng 0.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Lật cột theo hàng 0 rồi kiểm tra mỗi hàng đồng nhất (mô phỏng)

Bước 1: lật mọi cột `j` có `grid[0][j] = 1` ⇒ hàng 0 thành toàn 0. Bước 2: mỗi hàng khác phải toàn 0 hoặc toàn 1 (lật hàng đó). Cách này "làm thật" các phép lật trên một bản sao.

In [9]:
def simulate_flip(grid):
    g = [row[:] for row in grid]          # không sửa input
    n = len(g[0])
    for j in range(n):
        if g[0][j] == 1:
            for row in g:
                row[j] ^= 1
    return all(len(set(row)) == 1 for row in g)

for g, exp in examples:
    print(simulate_flip(g), "OK" if simulate_flip(g) == exp else "SAI")

True OK
False OK
True OK


**Độ phức tạp cách 2:** O(m·n) thời gian, O(m·n) bộ nhớ cho bản sao.

### 7.2 Cách 3 — Chuẩn hóa hàng + `set` (cách doocs)

Với mỗi hàng: nếu phần tử đầu của nó **khác** `grid[0][0]` thì lật cả hàng (lấy phần bù). Sau chuẩn hóa, mọi hàng phải **giống nhau** ⇔ tập các hàng chuẩn hóa có đúng 1 phần tử.

In [10]:
def normalize_set(grid):
    s = set()
    for row in grid:
        t = tuple(row) if row[0] == grid[0][0] else tuple(x ^ 1 for x in row)
        s.add(t)
    return len(s) == 1

for g, exp in examples:
    print(normalize_set(g), "OK" if normalize_set(g) == exp else "SAI")

True OK
False OK
True OK


In [11]:
class SolutionSet:
    def removeOnes(self, grid: List[List[int]]) -> bool:
        s = set()
        for row in grid:
            t = tuple(row) if row[0] == grid[0][0] else tuple(x ^ 1 for x in row)
            s.add(t)
        return len(s) == 1

print([SolutionSet().removeOnes(g) for g, _ in examples])

[True, False, True]


**Độ phức tạp cách 3:** O(m·n) thời gian (tạo tuple + băm), O(m·n) bộ nhớ worst case cho `set` (nếu sai, có thể có tới m mẫu khác nhau). Có thể dừng sớm khi `len(s) > 1`.

### 7.3 Cách 4 — So với hàng 0 hoặc phần bù (O(n) bộ nhớ, dừng sớm)

Không cần `set`: tính sẵn `first = grid[0]` và `comp = phần bù`. Mỗi hàng phải bằng một trong hai. So sánh list trong Python được làm bằng C nên rất nhanh.

In [12]:
def compare_first(grid):
    first = grid[0]
    comp = [x ^ 1 for x in first]
    return all(row == first or row == comp for row in grid)

for g, exp in examples:
    print(compare_first(g), "OK" if compare_first(g) == exp else "SAI")

True OK
False OK
True OK


### 7.4 Cách 5 — Kiểm tra chẵn lẻ bốn góc (công thức thuần)

Trực tiếp từ Insight 3: `grid[i][j] ^ grid[i][0] ^ grid[0][j] ^ grid[0][0] == 0` với mọi `i, j`.

In [13]:
def parity_check(grid):
    a00 = grid[0][0]
    g0 = grid[0]
    for row in grid:
        ri = row[0] ^ a00
        for j, x in enumerate(row):
            if x ^ ri ^ g0[j]:
                return False
    return True

for g, exp in examples:
    print(parity_check(g), "OK" if parity_check(g) == exp else "SAI")

True OK
False OK
True OK


**Độ phức tạp cách 4 và 5:** O(m·n) thời gian, O(n) (cách 4) hoặc O(1) (cách 5) bộ nhớ phụ.

## 8. TRACE từng bước

Trace cách 2 (lật theo hàng 0) trên ví dụ 1 và ví dụ 2, in ma trận sau mỗi lần lật cột và kết luận từng hàng.

In [14]:
def trace_simulate(grid):
    g = [row[:] for row in grid]
    print("ban đầu:"); show(g)
    for j in range(len(g[0])):
        if g[0][j] == 1:
            for row in g: row[j] ^= 1
            print(f"g[0][{j}] = 1 -> lật cột {j}:"); show(g)
    ok = True
    for i, row in enumerate(g):
        kind = "toàn 0 (không lật hàng)" if set(row) == {0} else "toàn 1 (lật hàng)" if set(row) == {1} else "LẪN LỘN -> thất bại"
        print(f"hàng {i}: {row} -> {kind}")
        ok &= len(set(row)) == 1
    print("Kết quả:", ok); return ok

trace_simulate(examples[0][0])
print()
trace_simulate(examples[1][0])

ban đầu:
   0 1 0
   1 0 1
   0 1 0
g[0][1] = 1 -> lật cột 1:
   0 0 0
   1 1 1
   0 0 0
hàng 0: [0, 0, 0] -> toàn 0 (không lật hàng)
hàng 1: [1, 1, 1] -> toàn 1 (lật hàng)
hàng 2: [0, 0, 0] -> toàn 0 (không lật hàng)
Kết quả: True

ban đầu:
   1 1 0
   0 0 0
   0 0 0
g[0][0] = 1 -> lật cột 0:
   0 1 0
   1 0 0
   1 0 0
g[0][1] = 1 -> lật cột 1:
   0 0 0
   1 1 0
   1 1 0
hàng 0: [0, 0, 0] -> toàn 0 (không lật hàng)
hàng 1: [1, 1, 0] -> LẪN LỘN -> thất bại
hàng 2: [1, 1, 0] -> LẪN LỘN -> thất bại
Kết quả: False


False

Trace cách 3 (chuẩn hóa) — in dạng chuẩn của từng hàng và tập `s`:

| hàng | nội dung | đầu hàng = grid[0][0]? | dạng chuẩn |
|---|---|---|---|
| 0 | 0 1 0 | có | (0,1,0) |
| 1 | 1 0 1 | không ⇒ lật | (0,1,0) |
| 2 | 0 1 0 | có | (0,1,0) |

`s = {(0,1,0)}` ⇒ `True`.

In [15]:
def trace_normalize(grid):
    s = set()
    for i, row in enumerate(grid):
        same = row[0] == grid[0][0]
        t = tuple(row) if same else tuple(x ^ 1 for x in row)
        s.add(t)
        print(f"hàng {i}: {row}, đầu hàng {'=' if same else '!='} grid[0][0] -> dạng chuẩn {t}, |s| = {len(s)}")
    print("Kết quả:", len(s) == 1)

trace_normalize(examples[0][0]); print()
trace_normalize(examples[1][0])

hàng 0: [0, 1, 0], đầu hàng = grid[0][0] -> dạng chuẩn (0, 1, 0), |s| = 1
hàng 1: [1, 0, 1], đầu hàng != grid[0][0] -> dạng chuẩn (0, 1, 0), |s| = 1
hàng 2: [0, 1, 0], đầu hàng = grid[0][0] -> dạng chuẩn (0, 1, 0), |s| = 1
Kết quả: True

hàng 0: [1, 1, 0], đầu hàng = grid[0][0] -> dạng chuẩn (1, 1, 0), |s| = 1
hàng 1: [0, 0, 0], đầu hàng != grid[0][0] -> dạng chuẩn (1, 1, 1), |s| = 2
hàng 2: [0, 0, 0], đầu hàng != grid[0][0] -> dạng chuẩn (1, 1, 1), |s| = 2
Kết quả: False


## 9. Code nộp LeetCode

Chọn cách 4 (so với hàng 0 hoặc phần bù): ngắn, dừng sớm, bộ nhớ O(n).

In [16]:
from typing import List

class Solution:
    def removeOnes(self, grid: List[List[int]]) -> bool:
        first = grid[0]
        comp = [1 - x for x in first]          # phần bù của hàng 0
        for row in grid:
            if row != first and row != comp:
                return False
        return True

In [17]:
sol = Solution()
for g, exp in examples:
    got = sol.removeOnes(g)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

True | kỳ vọng True | OK
False | kỳ vọng False | OK
True | kỳ vọng True | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (rút gọn):** mọi chuỗi thao tác tương đương với một cặp `(r, c) ∈ {0,1}^m × {0,1}^n`, với ô `(i, j)` cuối cùng bằng `grid[i][j] ⊕ r_i ⊕ c_j`. (XOR giao hoán/kết hợp và `a ⊕ a = 0`.)

**Định lý:** làm được ⇔ mọi hàng bằng hàng 0 hoặc phần bù của hàng 0.

(⇒) Có `(r, c)` với `grid[i][j] = r_i ⊕ c_j`. Khi đó hàng `i` = `c ⊕ r_i·(1,…,1)` và hàng 0 = `c ⊕ r_0·(1,…,1)`. Nếu `r_i = r_0` thì hai hàng bằng nhau; ngược lại chúng bù nhau.

(⇐) Đặt `c = grid[0]`, `r_i = 0` nếu hàng `i` bằng hàng 0, `r_i = 1` nếu là phần bù. Kiểm tra: `r_i ⊕ c_j` = `grid[0][j]` (nếu `r_i = 0`) = `grid[i][j]`; hoặc `1 ⊕ grid[0][j]` = `grid[i][j]` (nếu bù). ✔

Cách 3: hàng `i` có đầu hàng bằng `grid[0][0]` thì không thể là phần bù của hàng 0 (phần bù có đầu hàng khác) ⇒ phải bằng hàng 0; ngược lại phải là phần bù — chuẩn hóa đưa cả hai trường hợp về hàng 0, nên `|s| = 1` ⇔ điều kiện của định lý.

Cách 5: điều kiện bốn góc chính là `grid[i][j] = r_i ⊕ c_j` với `r_i = grid[i][0] ⊕ grid[0][0]`, `c_j = grid[0][j]`. ∎

## 11. Phân tích Time / Space Complexity

`m × n ≤ 300 × 300 = 90 000` ô.

| Cách | Time | Space phụ |
|---|---|---|
| 1a. Vét cạn hàng + cột | O(2^(m+n)·m·n) | O(m + n) |
| 1b. Vét cạn cột | O(2^n·m·n) | O(n) |
| 2. Lật theo hàng 0 | O(m·n) | O(m·n) (bản sao) |
| 3. Chuẩn hóa + set | O(m·n) | O(m·n) worst |
| 4. So với hàng 0 / bù | O(m·n) | O(n) |
| 5. Chẵn lẻ bốn góc | O(m·n) | O(1) |

Đo trên ma trận lớn nhất (trường hợp `True` — phải duyệt hết):

In [18]:
rng = random.Random(2128)
c = [rng.randint(0, 1) for _ in range(300)]
big = [[x ^ b for x in c] for b in (rng.randint(0, 1) for _ in range(300))]   # mọi hàng = c hoặc bù(c)
for name, f in [("simulate_flip", simulate_flip), ("normalize_set", normalize_set),
                ("compare_first", compare_first), ("parity_check", parity_check), ("Solution", Solution().removeOnes)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<14} -> {r}  {dt*1000:.2f} ms")

simulate_flip  -> True  2.38 ms
normalize_set  -> True  1.26 ms
compare_first  -> True  0.10 ms
parity_check   -> True  2.66 ms
Solution       -> True  0.11 ms


`compare_first` nhanh nhất vì phép so sánh list `row == first` chạy trong C.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute_all": brute_all, "brute_cols": brute_cols, "simulate_flip": simulate_flip,
       "normalize_set": normalize_set, "SolutionSet": SolutionSet().removeOnes,
       "compare_first": compare_first, "parity_check": parity_check, "Solution": Solution().removeOnes}
edge = examples + [
    ([[1]], True),                    # lật hàng hoặc cột
    ([[1, 1, 1]], True),              # một hàng: luôn True
    ([[0], [1], [1]], True),          # một cột: luôn True
    ([[1, 0], [0, 1]], True),         # bù nhau
    ([[1, 0], [0, 0]], False),
    ([[1, 1], [1, 1]], True),
    ([[0, 1, 1], [1, 0, 0], [0, 1, 1], [1, 0, 0]], True),
    ([[0, 1, 1], [1, 0, 0], [0, 1, 0]], False),
]
for g, exp in edge:
    for name, f in ALL.items():
        gc = [row[:] for row in g]
        assert f(gc) == exp, (name, g, exp)
        assert gc == g, (name, "đã sửa input")
print("Edge case OK:", len(edge))

Edge case OK: 11


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (m, n ≤ 4)

Ma trận hoàn toàn ngẫu nhiên hầu như luôn `False`, nên ta sinh một nửa số test theo dạng `r_i ⊕ c_j` (chắc chắn `True`) rồi thỉnh thoảng làm nhiễu một ô.

In [20]:
def gen(rng):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    if rng.random() < 0.5:
        return [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    r = [rng.randint(0, 1) for _ in range(m)]; c = [rng.randint(0, 1) for _ in range(n)]
    g = [[r[i] ^ c[j] for j in range(n)] for i in range(m)]
    if rng.random() < 0.4:
        g[rng.randrange(m)][rng.randrange(n)] ^= 1
    return g

rng = random.Random(42); cnt = [0, 0]
for _ in range(2500):
    g = gen(rng)
    ref = brute_all(g); cnt[ref] += 1
    for name, f in ALL.items():
        assert f([row[:] for row in g]) == ref, (name, g)
print("2500 case OK; số case False/True:", cnt)

2500 case OK; số case False/True: [905, 1595]


### 12.3 Tính chất: lật ngẫu nhiên hàng/cột không đổi đáp án

In [21]:
rng = random.Random(9)
for _ in range(300):
    m, n = rng.randint(1, 30), rng.randint(1, 30)
    g = gen(rng) if rng.random() < 0.3 else [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    base = Solution().removeOnes(g)
    h = [row[:] for row in g]
    for _ in range(10):
        if rng.random() < 0.5: flip_row(h, rng.randrange(len(h)))
        else: flip_col(h, rng.randrange(len(h[0])))
    assert Solution().removeOnes(h) == base
print("bất biến dưới phép lật: OK")

bất biến dưới phép lật: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra "mỗi hàng đồng nhất" hoặc "mỗi cột đồng nhất" mà không lật trước

In [22]:
def wrong_uniform_rows(grid):
    return all(len(set(row)) == 1 for row in grid)    # SAI: quên lật cột

print("bản sai :", wrong_uniform_rows(examples[0][0]))
print("bản đúng:", Solution().removeOnes(examples[0][0]))

bản sai : False
bản đúng: True


**Vì sao sai:** ví dụ 1 không có hàng nào đồng nhất, nhưng sau khi lật cột giữa thì có. **Sửa:** chuẩn hóa theo hàng 0 trước (cách 2) hoặc so với hàng 0/phần bù.

### Lỗi 2 — Chỉ so mỗi hàng với hàng 0 (quên trường hợp phần bù)

In [23]:
def wrong_only_equal(grid):
    return all(row == grid[0] for row in grid)

print("bản sai :", wrong_only_equal(examples[0][0]))
print("bản đúng:", Solution().removeOnes(examples[0][0]))

bản sai : False
bản đúng: True


**Sửa:** hàng `i` có thể bằng **phần bù** của hàng 0 (khi đó lật hàng `i`).

### Lỗi 3 — Tham lam "gặp số 1 thì lật hàng của nó" (lặp vô hạn hoặc sai)

In [24]:
def wrong_greedy(grid, max_ops=50):
    g = [row[:] for row in grid]; ops = 0
    while ops < max_ops:
        ones = [(i, j) for i, row in enumerate(g) for j, x in enumerate(row) if x]
        if not ones: return True
        i, j = ones[0]
        flip_row(g, i); ops += 1            # SAI: luôn lật hàng, không bao giờ lật cột
    return False

print("bản sai :", wrong_greedy(examples[0][0]))
print("bản đúng:", Solution().removeOnes(examples[0][0]))

bản sai : False
bản đúng: True


**Vì sao sai:** tham lam cục bộ không có cơ sở; lật hàng 0 của ví dụ 1 tạo ra `1 0 1`, rồi lại lật lại… quẩn mãi. Phải nhìn **cấu trúc** (`r_i ⊕ c_j`). **Sửa:** dùng tiêu chí ở mục 6.

### Lỗi 4 — Chuẩn hóa theo `row[0] == 1` nhưng so với `tuple(grid[0])` chưa chuẩn hóa

In [25]:
def wrong_normalize(grid):
    base = tuple(grid[0])                         # SAI: hàng 0 chưa được chuẩn hóa
    for row in grid:
        t = tuple(row) if row[0] == 0 else tuple(x ^ 1 for x in row)
        if t != base:
            return False
    return True

g = [[1, 0], [0, 1]]
print("bản sai :", wrong_normalize(g))
print("bản đúng:", Solution().removeOnes(g))

bản sai : False
bản đúng: True


**Vì sao sai:** mọi hàng được chuẩn hóa về "bắt đầu bằng 0", nhưng `base` là hàng 0 gốc bắt đầu bằng 1. **Sửa:** chuẩn hóa nhất quán (cả hàng 0 cũng chuẩn hóa), hoặc chuẩn hóa theo `grid[0][0]` như doocs.

### Lỗi 5 — Sửa trực tiếp `grid` khi mô phỏng lật cột

In [26]:
def wrong_inplace(grid):
    for j in range(len(grid[0])):
        if grid[0][j] == 1:
            for row in grid: row[j] ^= 1
    return all(len(set(row)) == 1 for row in grid)

g = [row[:] for row in examples[0][0]]
print("kết quả:", wrong_inplace(g), "| nhưng grid đã bị sửa thành:", g)
print("gọi lại lần 2 trên cùng biến (vẫn ra True nhưng input đã khác gốc):", g != examples[0][0])

kết quả: True | nhưng grid đã bị sửa thành: [[0, 0, 0], [1, 1, 1], [0, 0, 0]]
gọi lại lần 2 trên cùng biến (vẫn ra True nhưng input đã khác gốc): True


**Sửa:** làm trên bản sao `[row[:] for row in grid]`, hoặc dùng các cách chỉ đọc (3, 4, 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lật hàng/cột", "bật/tắt" ⇒ XOR ⇒ **thứ tự không quan trọng, mỗi phép tối đa 1 lần**.
- Hệ `a[i][j] = r_i ⊕ c_j` (hoặc `r_i + c_j`) ⇒ **cố định một biến** (đối xứng cho phép), rồi các biến khác bị ép ⇒ chỉ cần kiểm tra.
- Tiêu chí "mọi hàng bằng hàng 0 hoặc bù của nó" = ma trận có **hạng ≤ 1** trên GF(2) sau khi cộng một hằng số (hiểu thêm nếu học đại số tuyến tính).
- Bài cùng họ: 1072 Flip Columns For Maximum Number of Equal Rows, 861 Score After Flipping Matrix, 2017, 1284 Minimum Number of Flips to Convert Binary Matrix to Zero Matrix (lật chữ thập — cần BFS/bitmask).

## 15. Biến thể của bài

### 15.1 Biến thể A — **Số phép lật tối thiểu** (hoặc −1)

Nếu làm được thì chỉ có đúng **hai** lời giải `(r, c)` và `(¬r, ¬c)` (vì `r_0` quyết định tất cả). Số phép = `|r| + |c|` hoặc `(m − |r|) + (n − |c|)`; lấy min.

In [27]:
def min_flips(grid):
    if not Solution().removeOnes(grid):
        return -1
    m, n = len(grid), len(grid[0])
    c = grid[0][:]
    r = [0 if row == grid[0] else 1 for row in grid]
    k = sum(r) + sum(c)
    return min(k, m + n - k)

def min_flips_brute(grid):
    m, n = len(grid), len(grid[0]); best = -1
    for r in product([0, 1], repeat=m):
        for c in product([0, 1], repeat=n):
            if all(grid[i][j] ^ r[i] ^ c[j] == 0 for i in range(m) for j in range(n)):
                k = sum(r) + sum(c)
                best = k if best == -1 else min(best, k)
    return best

print(min_flips(examples[0][0]), min_flips(examples[1][0]), min_flips([[1, 1], [1, 1]]))
rng = random.Random(151)
for _ in range(800):
    g = gen(rng)
    assert min_flips(g) == min_flips_brute(g)
print("khớp brute force")

2 -1 2


khớp brute force


### 15.2 Biến thể B — LC 1072: chỉ được lật **cột**, tối đa hóa số hàng đồng nhất

Hàng `i` và hàng `k` cùng đồng nhất được với một tập lật cột ⇔ chúng giống hoặc bù nhau. Chuẩn hóa mỗi hàng (theo phần tử đầu) và đếm mẫu phổ biến nhất.

In [28]:
from collections import Counter
def max_equal_rows_after_flips(matrix):
    cnt = Counter(tuple(x ^ row[0] for x in row) for row in matrix)
    return max(cnt.values())

def max_equal_rows_brute(matrix):
    n = len(matrix[0]); best = 0
    for c in product([0, 1], repeat=n):
        best = max(best, sum(1 for row in matrix if len({x ^ y for x, y in zip(row, c)}) == 1))
    return best

print(max_equal_rows_after_flips([[0, 1], [1, 1]]), max_equal_rows_after_flips([[0, 1], [1, 0]]),
      max_equal_rows_after_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]]))
rng = random.Random(152)
for _ in range(500):
    g = [[rng.randint(0, 1) for _ in range(rng.randint(1, 5))]]
    g += [[rng.randint(0, 1) for _ in range(len(g[0]))] for _ in range(rng.randint(0, 5))]
    assert max_equal_rows_after_flips(g) == max_equal_rows_brute(g)
print("khớp brute force")

1 2 2
khớp brute force


Nhận xét: bài 2128 chính là câu hỏi "LC 1072 có đạt `m` (mọi hàng) không?".

### 15.3 Biến thể C — Đếm số ô 1 ít nhất có thể đạt được (không cần về 0 hoàn toàn)

Không còn tiêu chí đẹp; với `n` nhỏ (≤ 15) thử mọi tập lật cột, mỗi hàng chọn lật hay không để số 1 nhỏ nhất: `min(ones, n − ones)`. O(2^n·m·n).

In [29]:
def min_ones(grid):
    m, n = len(grid), len(grid[0])
    masks = [sum(b << j for j, b in enumerate(row)) for row in grid]
    best = m * n
    for c in range(1 << n):
        tot = 0
        for mk in masks:
            ones = bin(mk ^ c).count("1")
            tot += min(ones, n - ones)
        best = min(best, tot)
    return best

print(min_ones(examples[0][0]), min_ones(examples[1][0]), min_ones([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))

0 1 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn r, c | thử 2^(m+n) | O(2^(m+n)·mn) | O(m+n) | kiểm chứng, rất nhỏ |
| 1b. Vét cạn c | hàng tự quyết | O(2^n·mn) | O(n) | n nhỏ, biến thể tối ưu hóa |
| 2. Lật theo hàng 0 | mô phỏng thật | O(mn) | O(mn) | trực quan nhất |
| 3. Chuẩn hóa + set | mọi hàng về một mẫu | O(mn) | O(mn) | dễ nhớ, mở rộng sang đếm (1072) |
| 4. So với hàng 0/bù | tiêu chí trực tiếp | O(mn) | O(n) | **mặc định**, nhanh nhất trong Python |
| 5. Chẵn lẻ bốn góc | công thức XOR | O(mn) | O(1) | khi cần O(1) bộ nhớ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `removeOnes([[1,0,1],[0,1,0]])` và `removeOnes([[1,0,1],[0,1,1]])`?

**Bài 2 (tính tay):** Với `grid = [[0,1,0],[1,0,1],[0,1,0]]`, hãy viết ra **cả hai** cặp `(r, c)` là lời giải. Mỗi cặp dùng bao nhiêu phép lật?

**Bài 3 (điền TODO):** Viết `bt3_todo(grid)` trả về **danh sách thao tác** dạng `[("row", i), ("col", j), ...]` làm `grid` về 0 (bất kỳ lời giải nào), hoặc `None` nếu không thể.

**Bài 4 (tự viết lại):** Viết `bt4_todo(grid)` dùng cách 5 (chẵn lẻ bốn góc) nhưng duyệt theo **cột** trước (vòng ngoài là `j`).

**Bài 5 (mini challenge):** Mỗi ô có thể là 0, 1, hoặc 2 và phép "lật" là cộng 1 modulo 3 cho cả hàng/cột (được làm nhiều lần). Viết `bt5_todo(grid)` kiểm tra có thể đưa mọi ô về 0 không.

In [30]:
def bt3_todo(grid):
    # TODO: trả về list thao tác ("row", i)/("col", j), hoặc None
    pass

def bt4_todo(grid):
    # TODO: kiểm tra chẵn lẻ bốn góc, vòng ngoài theo cột
    pass

def bt5_todo(grid):
    # TODO: phiên bản modulo 3
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: so hàng 1 với hàng 0. Mức 2: `010` là phần bù của `101`; `011` thì không. Mức 3: `True`, `False`.

**Bài 2.** Mức 1: `r_0 = 0` ⇒ `c = hàng 0`. Mức 2: `c = (0,1,0)`, `r = (0,1,0)`. Mức 3: lời giải còn lại `r = (1,0,1)`, `c = (1,0,1)`; số phép lần lượt 2 và 4.

**Bài 3.** Mức 1: kiểm tra trước bằng `Solution`. Mức 2: lật cột `j` với `grid[0][j] = 1`. Mức 3: sau đó lật hàng `i` nếu hàng `i` (gốc) khác hàng 0.

**Bài 4.** Mức 1: điều kiện không phụ thuộc thứ tự duyệt. Mức 2: với mỗi `j`, `cj = grid[0][j] ^ grid[0][0]`. Mức 3: kiểm tra `grid[i][j] == grid[i][0] ^ cj` với mọi `i`.

**Bài 5.** Mức 1: phương trình thành `grid[i][j] + r_i + c_j ≡ 0 (mod 3)`, `r_i, c_j ∈ {0,1,2}`. Mức 2: có thể cố định `r_0 = 0` (cộng `t` vào mọi hàng và trừ `t` ở mọi cột là không đổi). Mức 3: `c_j = −grid[0][j]`, `r_i = −grid[i][0] − c_0`, rồi kiểm tra mọi ô: `grid[i][j] − grid[i][0] − grid[0][j] + grid[0][0] ≡ 0 (mod 3)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `True`, `False`. **Bài 2:** `(r, c) = ((0,1,0), (0,1,0))` với 2 phép, và `((1,0,1), (1,0,1))` với 4 phép.

In [31]:
assert Solution().removeOnes([[1, 0, 1], [0, 1, 0]]) is True
assert Solution().removeOnes([[1, 0, 1], [0, 1, 1]]) is False
g = examples[0][0]
for r, c in [((0, 1, 0), (0, 1, 0)), ((1, 0, 1), (1, 0, 1))]:
    assert all(g[i][j] ^ r[i] ^ c[j] == 0 for i in range(3) for j in range(3))
assert min_flips(g) == 2

def bt3(grid):
    if not Solution().removeOnes(grid): return None
    ops = [("col", j) for j, x in enumerate(grid[0]) if x == 1]
    ops += [("row", i) for i, row in enumerate(grid) if row != grid[0]]
    return ops

def bt4(grid):
    m, n = len(grid), len(grid[0])
    for j in range(n):
        cj = grid[0][j] ^ grid[0][0]
        for i in range(m):
            if grid[i][j] != grid[i][0] ^ cj:
                return False
    return True

def bt5(grid):
    a00 = grid[0][0]; g0 = grid[0]
    return all((row[j] - row[0] - g0[j] + a00) % 3 == 0 for row in grid for j in range(len(row)))

def bt5_brute(grid):
    m, n = len(grid), len(grid[0])
    for r in product(range(3), repeat=m):
        for c in product(range(3), repeat=n):
            if all((grid[i][j] + r[i] + c[j]) % 3 == 0 for i in range(m) for j in range(n)):
                return True
    return False

rng = random.Random(1928)
for _ in range(1500):
    g = gen(rng)
    ref = brute_all(g)
    assert bt4(g) == ref
    ops = bt3(g)
    assert (ops is not None) == ref
    if ops is not None:
        h = [row[:] for row in g]
        for t, k in ops: (flip_row if t == "row" else flip_col)(h, k)
        assert all(x == 0 for row in h for x in row)
for _ in range(400):
    m, n = rng.randint(1, 3), rng.randint(1, 3)
    if rng.random() < 0.5:
        r = [rng.randrange(3) for _ in range(m)]; c = [rng.randrange(3) for _ in range(n)]
        g = [[(-(r[i] + c[j])) % 3 for j in range(n)] for i in range(m)]
    else:
        g = [[rng.randrange(3) for _ in range(n)] for _ in range(m)]
    assert bt5(g) == bt5_brute(g)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# grid về 0 bằng lật hàng/cột  <=>  mọi hàng == hàng 0 hoặc == bù(hàng 0)
first = grid[0]
comp = [1 - x for x in first]
return all(row == first or row == comp for row in grid)
```

**Checklist:**
- [ ] Hiểu: mỗi hàng/cột lật tối đa 1 lần, thứ tự không quan trọng.
- [ ] Xét **cả** trường hợp "giống" và "bù".
- [ ] Không sửa `grid` gốc nếu mô phỏng.
- [ ] Ma trận 1 hàng hoặc 1 cột ⇒ luôn `True`.
- [ ] (Premium) Đối chiếu lại đề gốc nếu có tài khoản.